In [1]:
import pandas as pd
import yake_helper_funcs as yhf
from sklearn.cluster import SpectralClustering
import numpy as np
import itertools

forum_posts = pd.read_csv("../input/meta-kaggle/ForumMessages.csv")

# get forum posts

# subsample forum posts
sample_posts = forum_posts.Message[-1000:].astype(str)

# Extract set of keywords from each post

In [2]:
# extact keywords & tokenize
keywords = yhf.keywords_yake(sample_posts)
keywords_tokenized = yhf.tokenizing_after_YAKE(keywords)
keyword_sets = [set(post) for post in keywords_tokenized]


/opt/conda/lib/python3.6/site-packages/yake/datarepresentation.py:106: RuntimeWarning: Mean of empty slice.
  avgTF = validTFs.mean()
/opt/conda/lib/python3.6/site-packages/numpy/core/_methods.py:85: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.6/site-packages/numpy/core/_methods.py:140: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/opt/conda/lib/python3.6/site-packages/numpy/core/_methods.py:110: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/opt/conda/lib/python3.6/site-packages/numpy/core/_methods.py:132: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [3]:

# remove empty sets
keyword_sets_noempty = [x for x in keyword_sets if x]

# Get word vectors for keywords in post

In [4]:
vectors = pd.read_csv("../input/fine-tuning-word2vec-2-0/kaggle_word2vec.model", 
                      delim_whitespace=True,
                      skiprows=[0], 
                      header=None
                     )

# set words as index rather than first column
vectors.index = vectors[0]
vectors.drop(0, axis=1, inplace=True)

/opt/conda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3057: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [5]:
def vectors_from_post(post):
    all_words = [] 

    for words in post:
        all_words.append(words) 
        
    return(vectors[vectors.index.isin(all_words)])


def doc_embed_from_post(post):
    test_vectors = vectors_from_post(post)

    return(test_vectors.mean())

In [6]:
# get document embeddings for 100 posts
num_of_posts = 100
doc_embeddings = np.zeros([num_of_posts, 300])


# TODO: handle posts where all words out OOV
for i in range(num_of_posts):
    embeddings = np.array(doc_embed_from_post(keyword_sets[i]))
    if np.isnan(embeddings).any():
        doc_embeddings[i,:] = np.zeros([1,300])
    else:
        doc_embeddings[i,:] = embeddings

# Clustering!

In [7]:
# note that (default) k-means label assignment didn't work well
clustering = SpectralClustering(assign_labels="discretize").fit(doc_embeddings)

In [8]:
# look at the labels for each of the posts
clustering.labels_

array([1, 1, 0, 0, 7, 0, 1, 1, 0, 6, 1, 7, 5, 0, 0, 1, 1, 0, 0, 1, 3, 7,
       1, 1, 1, 5, 1, 0, 1, 1, 6, 0, 1, 1, 7, 5, 1, 1, 0, 0, 4, 0, 1, 1,
       0, 0, 0, 1, 0, 1, 3, 1, 1, 1, 5, 5, 3, 2, 0, 1, 0, 3, 1, 0, 5, 1,
       1, 1, 2, 1, 1, 1, 5, 1, 2, 0, 1, 1, 1, 1, 1, 4, 1, 7, 4, 0, 3, 1,
       2, 0, 1, 4, 2, 0, 0, 7, 7, 0, 1, 0])

In [9]:
# explore our posts by cluster
post_subset = keyword_sets[0:num_of_posts]

def get_keyword_set_by_cluster(number):
    cluster_index = list(clustering.labels_ == number)
    return(list(itertools.compress(post_subset, cluster_index)))

In [10]:
# this cluster looks like it's out of vocabular
get_keyword_set_by_cluster(3)

[{'fantastic', 'work'},
 {'beautiful', 'contributing', 'keep', 'vikumsw', 'work'},
 {'aleksandradeis', 'great', 'work'},
 {'data', 'identify', 'million', 'rows', 'set'},
 {'colab', 'good', 'google', 'work'}]

In [11]:
# this cluster looks like it's about deep learning
get_keyword_set_by_cluster(6)

[{'effort', 'worth'}, {'bookmark', 'worth'}]